# Predicting Student Test Scores 
## Score: 8.71001 

In [1]:
import os
import hashlib
import numpy as np
import pandas as pd
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [2]:
train_df = pd.read_csv('playground-series-s6e1/train.csv')
test_df = pd.read_csv('playground-series-s6e1/test.csv')

ORIGINAL_PATH = 'Exam_Score_Prediction.csv'
if not os.path.exists(ORIGINAL_PATH):
    raise FileNotFoundError(f'Original dataset not found at {ORIGINAL_PATH}. Download from https://www.kaggle.com/datasets/lainguyn123/student-performance-factors')

original_df = pd.read_csv(ORIGINAL_PATH)

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
print(f'Original shape: {original_df.shape}')

TARGET = 'exam_score'
ID_COL = 'id'

Train shape: (630000, 13)
Test shape: (270000, 12)
Original shape: (20000, 13)


In [3]:
def engineer_features(df):
    df = df.copy()
    
    ordinal_maps = {
        'sleep_quality': {'poor': 0, 'average': 1, 'good': 2},
        'facility_rating': {'low': 0, 'medium': 1, 'high': 2},
        'exam_difficulty': {'easy': 0, 'moderate': 1, 'hard': 2},
        'internet_access': {'no': 0, 'yes': 1}
    }

    for col, mapping in ordinal_maps.items():
        if col in df.columns:
            df[f'{col}_ord'] = df[col].map(mapping).fillna(1)
    
    df['study_x_attendance'] = df['study_hours'] * df['class_attendance']
    df['study_x_sleep'] = df['study_hours'] * df['sleep_hours']
    df['study_x_sleep_quality'] = df['study_hours'] * df['sleep_quality_ord']
    df['attendance_x_facility'] = df['class_attendance'] * df['facility_rating_ord']
    df['age_x_study'] = df['age'] * df['study_hours']
    
    df['study_per_sleep'] = df['study_hours'] / (df['sleep_hours'] + 1)
    df['attendance_per_study'] = df['class_attendance'] / (df['study_hours'] + 1)
    df['efficiency'] = df['study_x_attendance'] / (df['sleep_hours'] + 1)
    
    df['age_squared'] = df['age'] ** 2
    df['study_squared'] = df['study_hours'] ** 2
    df['attendance_squared'] = df['class_attendance'] ** 2
    
    df['log_study'] = np.log1p(df['study_hours'])
    df['log_attendance'] = np.log1p(df['class_attendance'])

    df['sleep_dist_8h'] = np.abs(df['sleep_hours'] - 8)
    df['attendance_dist_100'] = np.abs(df['class_attendance'] - 100)
    
    df['engagement_score'] = (df['study_hours'] * 10 + 
                              df['class_attendance'] + 
                              df['sleep_quality_ord'] * 15)
    
    return df

train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)
original_fe = engineer_features(original_df)

print(f'Features after engineering: {len([c for c in train_fe.columns if c not in [TARGET, ID_COL]])}')

Features after engineering: 31


In [4]:
cat_cols = ['gender', 'course', 'study_method', 'sleep_quality', 
            'facility_rating', 'exam_difficulty', 'internet_access']
cat_cols = [c for c in cat_cols if c in train_fe.columns]

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    train_fe[col] = le.fit_transform(train_fe[col].astype(str))
    test_fe[col] = le.transform(test_fe[col].astype(str))
    original_fe[col] = le.transform(original_fe[col].astype(str))
    le_dict[col] = le

feature_cols = [c for c in train_fe.columns if c not in [TARGET, ID_COL, 'student_id']]

X_train = train_fe[feature_cols].fillna(0).values
y_train = train_fe[TARGET].values
X_test = test_fe[feature_cols].fillna(0).values
X_orig = original_fe[feature_cols].fillna(0).values
y_orig = original_fe[TARGET].values
test_ids = test_df[ID_COL].values

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'X_original shape: {X_orig.shape}')

X_train shape: (630000, 31)
X_test shape: (270000, 31)
X_original shape: (20000, 31)


In [5]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))
oof_rf = np.zeros(len(X_train))

test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_rf = np.zeros(len(X_test))

fold_scores = {'LightGBM': [], 'XGBoost': [], 'RandomForest': []}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
    print(f'\n{"="*60}')
    print(f'FOLD {fold}/{N_FOLDS}')
    print(f'{"="*60}')
    
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
    X_tr_comb = np.vstack([X_tr, X_orig])
    y_tr_comb = np.concatenate([y_tr, y_orig])
    
    print('  Training LightGBM...')
    lgb_train = lgb.Dataset(X_tr_comb, y_tr_comb)
    lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

    lgb_params = {
        'objective': 'rmse',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': 0.03,
        'num_leaves': 31,
        'max_depth': 7,
        'min_child_samples': 20,
        'subsample': 0.8,
        'subsample_freq': 1,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.3,
        'reg_lambda': 0.3,
        'random_state': 42,
        'verbose': -1,
        'n_jobs': -1
    }
    
    lgb_model = lgb.train(
        lgb_params,
        lgb_train,
        num_boost_round=2000,
        valid_sets=[lgb_val],
        callbacks=[lgb.early_stopping(100, verbose=False)])

    oof_lgb[val_idx] = lgb_model.predict(X_val)
    test_lgb += lgb_model.predict(X_test) / N_FOLDS
    
    rmse_lgb = np.sqrt(mean_squared_error(y_val, oof_lgb[val_idx]))
    fold_scores['LightGBM'].append(rmse_lgb)
    print(f'    LightGBM RMSE: {rmse_lgb:.5f}')

    print('  Training XGBoost...')
    dtrain = xgb.DMatrix(X_tr_comb, y_tr_comb)
    dval = xgb.DMatrix(X_val, y_val)
    
    xgb_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'learning_rate': 0.03,
        'max_depth': 7,
        'min_child_weight': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.3,
        'reg_lambda': 1.0,
        'random_state': 42,
        'tree_method': 'hist',
        'verbosity': 0
    }

    xgb_model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=2000,
        evals=[(dval, 'val')],
        early_stopping_rounds=100,
        verbose_eval=False
    )
    
    oof_xgb[val_idx] = xgb_model.predict(dval)
    test_xgb += xgb_model.predict(xgb.DMatrix(X_test)) / N_FOLDS
    
    rmse_xgb = np.sqrt(mean_squared_error(y_val, oof_xgb[val_idx]))
    fold_scores['XGBoost'].append(rmse_xgb)
    print(f'    XGBoost RMSE: {rmse_xgb:.5f}')

    print('  Training Random Forest...')
    rf_model = RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    
    rf_model.fit(X_tr_comb, y_tr_comb)
    
    oof_rf[val_idx] = rf_model.predict(X_val)
    test_rf += rf_model.predict(X_test) / N_FOLDS
    
    rmse_rf = np.sqrt(mean_squared_error(y_val, oof_rf[val_idx]))
    fold_scores['RandomForest'].append(rmse_rf)
    print(f'    RandomForest RMSE: {rmse_rf:.5f}')

rmse_lgb_oof = np.sqrt(mean_squared_error(y_train, oof_lgb))
rmse_xgb_oof = np.sqrt(mean_squared_error(y_train, oof_xgb))
rmse_rf_oof = np.sqrt(mean_squared_error(y_train, oof_rf))

print(f'\n{"="*60}')
print('OOF SCORES:')
print(f'  LightGBM:      {rmse_lgb_oof:.6f}')
print(f'  XGBoost:       {rmse_xgb_oof:.6f}')
print(f'  Random Forest: {rmse_rf_oof:.6f}')
print(f'{"="*60}')


FOLD 1/5
  Training LightGBM...
    LightGBM RMSE: 8.74266
  Training XGBoost...
    XGBoost RMSE: 8.73586
  Training Random Forest...
    RandomForest RMSE: 8.91945

FOLD 2/5
  Training LightGBM...
    LightGBM RMSE: 8.74975
  Training XGBoost...
    XGBoost RMSE: 8.73920
  Training Random Forest...
    RandomForest RMSE: 8.92298

FOLD 3/5
  Training LightGBM...
    LightGBM RMSE: 8.74040
  Training XGBoost...
    XGBoost RMSE: 8.73057
  Training Random Forest...
    RandomForest RMSE: 8.91867

FOLD 4/5
  Training LightGBM...
    LightGBM RMSE: 8.76152
  Training XGBoost...
    XGBoost RMSE: 8.75001
  Training Random Forest...
    RandomForest RMSE: 8.92136

FOLD 5/5
  Training LightGBM...
    LightGBM RMSE: 8.77714
  Training XGBoost...
    XGBoost RMSE: 8.76285
  Training Random Forest...
    RandomForest RMSE: 8.95426

OOF SCORES:
  LightGBM:      8.754304
  XGBoost:       8.743707
  Random Forest: 8.927355


In [6]:
def ensemble_rmse(weights, preds_list, y_true):
    weights = np.abs(weights) / np.sum(np.abs(weights))
    ensemble_pred = sum(w * p for w, p in zip(weights, preds_list))
    return np.sqrt(mean_squared_error(y_true, ensemble_pred))

preds_list = [oof_lgb, oof_xgb, oof_rf]
test_preds_list = [test_lgb, test_xgb, test_rf]

result = minimize(
    ensemble_rmse,
    x0=[0.33, 0.33, 0.34],
    args=(preds_list, y_train),
    bounds=[(0.15, 0.5), (0.15, 0.5), (0.15, 0.5)],
    method='L-BFGS-B'
)

weights = np.abs(result.x) / np.sum(np.abs(result.x))

print(f'OPTIMAL WEIGHTS:')
print(f'  LightGBM:      {weights[0]:.4f} ({weights[0]*100:.1f}%)')
print(f'  XGBoost:       {weights[1]:.4f} ({weights[1]*100:.1f}%)')
print(f'  Random Forest: {weights[2]:.4f} ({weights[2]*100:.1f}%)')

oof_ensemble = sum(w * p for w, p in zip(weights, preds_list))
test_ensemble = sum(w * p for w, p in zip(weights, test_preds_list))

rmse_ensemble = np.sqrt(mean_squared_error(y_train, oof_ensemble))

print(f'\nFINAL RESULTS:')
print(f'  LightGBM:      {rmse_lgb_oof:.6f}')
print(f'  XGBoost:       {rmse_xgb_oof:.6f}')
print(f'  Random Forest: {rmse_rf_oof:.6f}')
print(f'  ENSEMBLE:      {rmse_ensemble:.6f}')
print(f'  Improvement:   {min(rmse_lgb_oof, rmse_xgb_oof, rmse_rf_oof) - rmse_ensemble:.6f}')

OPTIMAL WEIGHTS:
  LightGBM:      0.4348 (43.5%)
  XGBoost:       0.4348 (43.5%)
  Random Forest: 0.1304 (13.0%)

FINAL RESULTS:
  LightGBM:      8.754304
  XGBoost:       8.743707
  Random Forest: 8.927355
  ENSEMBLE:      8.743678
  Improvement:   0.000029


In [7]:
test_ensemble_clipped = np.clip(test_ensemble, 0, 100)

submission = pd.DataFrame({'id': test_ids, 'exam_score': test_ensemble_clipped})
submission.to_csv('submission.csv', index=False)

with open('submission.csv', 'rb') as f:
    md5 = hashlib.md5(f.read()).hexdigest()

print('submission.csv')
print('md5', md5)
print(f'pred_mean {test_ensemble_clipped.mean():.4f}')
print(f'pred_std {test_ensemble_clipped.std():.4f}')
print(f'pred_min {test_ensemble_clipped.min():.4f}')
print(f'pred_max {test_ensemble_clipped.max():.4f}')

submission.csv
md5 46ac5ff97de5cdbd9f5e39499fe42117
pred_mean 62.5430
pred_std 16.7277
pred_min 15.9418
pred_max 100.0000
